# Credit Card ETL — Medallion Pipeline

Source: Kaggle `jinquan/cc-sample-data` → `cc_sample_transaction.json`

| Layer | What happens | Output |
|---|---|---|
| **Bronze** | Land the raw JSON untouched, add ingestion lineage | `data/bronze/transactions` |
| **Silver** | Flatten nested JSON, type, clean names, dedup, DQ gate, encrypt + hash PII | `data/silver/transactions` |
| **Gold** |  Aggregates for visualisation + one restricted table that decrypts PII | `data/gold/*` |

Run order is top to bottom. Each layer reads the layer above from disk, so a
layer can be re-run on its own.

## 0. Setup

In [1]:
# All dependencies are pinned in requirements.txt 
#
# This notebook runs in a dedicated venv. Select the "Python (cc-fraud-etl)" kernel. To rebuild the environment from scratch:
#
#   python3 -m venv .venv
#   ./.venv/bin/python -m pip install -r requirements.txt
#   ./.venv/bin/python -m ipykernel install --user \
#       --name cc-fraud-etl --display-name "Python (cc-fraud-etl)"

# Confirm the kernel is the venv, not base:
import sys
print(sys.prefix)

/Users/lewisyuan/Documents/Github_RnD/de-zoomcamp-lewis/week-2/ny_taxi_postgres_data/.venv


In [2]:
import os
from pyspark.sql import SparkSession, functions as F, Window

# Where each medallion layer lands.
DATA_ROOT   = "data"
BRONZE_PATH = f"{DATA_ROOT}/bronze/transactions"
SILVER_PATH = f"{DATA_ROOT}/silver/transactions"
GOLD_PATH   = f"{DATA_ROOT}/gold"
DQ_RESULTS  = f"{DATA_ROOT}/_dq_results"
DQ_RULES    = "dq_rules.yaml"

In [3]:
spark = (SparkSession.builder.master("local[2]").appName("paynet-test")
         .config("spark.ui.enabled", "false")
         .config("spark.sql.shuffle.partitions", "4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Target timezone is UTC+8 — Spark stores an instant and renders in session tz.
spark.conf.set("spark.sql.session.timeZone", "Asia/Kuala_Lumpur")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/20 18:27:02 WARN Utils: Your hostname, Lewiss-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.229.117.164 instead (on interface en0)
26/09/20 18:27:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 18:27:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


---
# 1. Bronze — Landing layer


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jinquan/cc-sample-data")

print("Path to dataset files:", path)
print(os.listdir(path))   # see what files came down

/Users/lewisyuan/Documents/Github_RnD/de-zoomcamp-lewis/week-2/ny_taxi_postgres_data/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/lewisyuan/.cache/kagglehub/datasets/jinquan/cc-sample-data/versions/1
['cc_sample_transaction.json']


In [5]:
bronze = spark.read.json(os.path.join(path, "cc_sample_transaction.json"))



In [6]:
bronze.printSchema()
bronze.show(5)

root
 |-- Unnamed: 0: string (nullable = true)
 |-- amt: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cc_bic: string (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- is_fraud: string (nullable = true)
 |-- merch_eff_time: string (nullable = true)
 |-- merch_last_update_time: string (nullable = true)
 |-- merch_lat: string (nullable = true)
 |-- merch_long: string (nullable = true)
 |-- merch_zipcode: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- personal_detail: string (nullable = true)
 |-- trans_date_trans_time: string (nullable = true)
 |-- trans_num: string (nullable = true)

+----------+------+-------------+-----------+----------------+--------+----------------+----------------------+------------------+-----------+-------------+--------------------+--------------------+---------------------+--------------------+
|Unnamed: 0|   amt|     category|     cc_bic|          cc_num|is_fraud|  merch_eff_time|merch_last_update_time

In [7]:
# Lineage columns — easy to trace files and ingestion time, in case there is file patching from source.
bronze = (bronze
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.input_file_name()))

bronze.write.mode("overwrite").parquet(BRONZE_PATH)

### Data checking before transformation

In [8]:
# Epoch columns arrive as strings with inconsistent digit widths. Suggested by AI.
bronze.select(
    F.length("merch_eff_time").alias("len_eff"),
    F.length("merch_last_update_time").alias("len_upd")
).distinct().show()

+-------+-------+
|len_eff|len_upd|
+-------+-------+
|     15|     13|
|     13|     12|
|     11|     13|
|     12|     12|
|     15|     11|
|     13|     13|
|     14|     12|
|     14|     11|
|     16|     13|
|     16|     11|
|     13|     11|
|     16|     12|
|     15|     12|
|     12|     13|
|     14|     13|
|     12|     11|
+-------+-------+



In [9]:
# Quick check personal_detail column before flattening and cleanup. 
bronze.select("personal_detail").show(10, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|personal_detail                                                                                                                                                                                                                                                                      |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"person_name":"Jennifer,Banks,eeeee","gender":"F","address":"{\"street\":\"561 Perry Cove\",\"city\":\"Moravian Falls\",\"state\":\"NC\",\"zip\":\"28654\"}","

---
# 2. Silver — Transformation layer

Denormalization , correct the data type, dedup and dq check

In [10]:
silver = spark.read.parquet(BRONZE_PATH)

### 2.1 Timestamps

Why using different timestamp based on different length 

In [11]:
silver = (silver
    .withColumn("merch_eff_time",
                F.timestamp_micros(F.col("merch_eff_time").cast("long")))
    .withColumn("merch_last_update_time",
                F.timestamp_millis(F.col("merch_last_update_time").cast("long")))
    .withColumn("trans_date_trans_time",
                F.to_utc_timestamp(F.to_timestamp("trans_date_trans_time"), "UTC")
    )
)

### 2.2 Flatten the nested JSON

In [12]:
pd_schema = ("person_name string, gender string, address string, "
             "lat string, long string, city_pop string, job string, dob string")
addr_schema = "street string, city string, state string, zip string"

silver = (silver
    .withColumn("pd", F.from_json("personal_detail", pd_schema))
    .withColumn("addr", F.from_json(F.col("pd.address"), addr_schema)))

In [13]:
# To double check if there is null date inside the JSON
print("pd   unparsed:", silver.filter(F.col("pd").isNull() & F.col("personal_detail").isNotNull()).count())
print("addr unparsed:", silver.filter(F.col("addr").isNull() & F.col("pd.address").isNotNull()).count())

pd   unparsed: 0


addr unparsed: 0


In [14]:
silver = silver.select(
    # transaction-level columns already at top level
    "trans_num",
    "trans_date_trans_time",
    "cc_num",
    "cc_bic",
    "merchant",
    "category",
    F.col("amt").cast("decimal(12,2)").alias("amt"),
    F.col("is_fraud").cast("int").alias("is_fraud"),

    # merchant geo
    F.col("merch_lat").cast("double").alias("merch_lat"),
    F.col("merch_long").cast("double").alias("merch_long"),
    "merch_zipcode",

    # from personal_detail
    F.col("pd.person_name").alias("person_name"),
    F.col("pd.gender").alias("gender"),
    F.col("pd.job").alias("job"),
    F.to_date(F.col("pd.dob"), "yyyy-MM-dd").alias("dob"),
    F.col("pd.lat").cast("double").alias("cust_lat"),
    F.col("pd.long").cast("double").alias("cust_long"),
    F.col("pd.city_pop").cast("int").alias("city_pop"),

    # from the nested address
    F.col("addr.street").alias("street"),
    F.col("addr.city").alias("city"),
    F.col("addr.state").alias("state"),
    F.col("addr.zip").alias("zip"),

    # ADDED: carried from bronze — _ingested_at is the dedup tiebreaker in 2.4
    "_ingested_at",
    "_source_file",
)

### 2.3 Clean the corrupted names

In [15]:
JUNK_WORD = r"\b\w*(\w)\1{2,}\w*\b"        # any word with a 3+ identical-char run
NON_NAME  = r"[^A-Za-zÀ-ÿ\s'\-]"           # keep letters (incl. accents), space, ' and -

# CHANGED: was F.col("pd.person_name") — the `pd` struct no longer exists after
# the select in 2.2, so this reads the aliased flat column instead.
cleaned = F.regexp_replace(F.col("person_name"), JUNK_WORD, " ")
cleaned = F.regexp_replace(cleaned, NON_NAME, " ")      # , / @ | ; ! all become spaces

tokens = F.filter(
    F.split(F.trim(cleaned), r"\s+"),
    lambda c: F.length(c) > 0
)

silver = (silver
    .withColumn("name_tokens", tokens)
    .withColumn("first_name",
                F.when(F.size("name_tokens") > 0, F.element_at("name_tokens", 1)))
    .withColumn("last_name",
                F.when(F.size("name_tokens") > 1, F.element_at("name_tokens", -1))))

In [16]:
# Parse status keeps bad rows visible instead of silently nulling them.
silver = silver.withColumn("name_parse_status",
    F.when(F.col("person_name").isNull(), "null_source")
     .when(F.size("name_tokens") == 2, "ok")
     .when(F.size("name_tokens") == 0, "empty")
     .when(F.size("name_tokens") == 1, "single_token")
     .otherwise("multi_token"))

silver.groupBy("name_parse_status").count().show()

+-----------------+-------+
|name_parse_status|  count|
+-----------------+-------+
|               ok|1296675|
+-----------------+-------+



In [17]:
silver.select("person_name", "first_name", "last_name", "name_parse_status").show(10, truncate=False)

+------------------------+----------+---------+-----------------+
|person_name             |first_name|last_name|name_parse_status|
+------------------------+----------+---------+-----------------+
|Jennifer,Banks,eeeee    |Jennifer  |Banks    |ok               |
|Stephanie,Gill,eeeee    |Stephanie |Gill     |ok               |
|Edward@Sanchez          |Edward    |Sanchez  |ok               |
|Jeremy/White, !         |Jeremy    |White    |ok               |
|Tyler@Garcia            |Tyler     |Garcia   |ok               |
|Jennifer,Conner,eeeee   |Jennifer  |Conner   |ok               |
|Kelsey, , Richards NOOOO|Kelsey    |Richards |ok               |
|Steven, Williams        |Steven    |Williams |ok               |
|Heather, , Chase NOOOO  |Heather   |Chase    |ok               |
|Melissa@Aguilar         |Melissa   |Aguilar  |ok               |
+------------------------+----------+---------+-----------------+
only showing top 10 rows


### 2.4 Deduplicate

In [18]:
# Surrogate key: md5 of the business key (trans_num + transaction instant).
#
# The best case is unique key provided by backend such as transaction ID. 
# If not, use a combination of fields that uniquely identify a row. 
# Here we use trans_num and trans_date_trans_time.

KEY_COLS = [
    F.coalesce(F.col("trans_num"), F.lit("__NULL__")),
    F.coalesce(F.unix_micros(F.col("trans_date_trans_time")).cast("string"), F.lit("__NULL__")),
]

silver = silver.withColumn("_scd_unique_id", F.md5(F.concat_ws("||", *KEY_COLS)))

silver.select("trans_num", "trans_date_trans_time", "_scd_unique_id").show(5, truncate=False)



+--------------------------------+---------------------+--------------------------------+
|trans_num                       |trans_date_trans_time|_scd_unique_id                  |
+--------------------------------+---------------------+--------------------------------+
|0b242abb623afc578575680df30655b9|2019-01-01 00:00:18  |f19057cf555faa6138f8791ff801379b|
|1f76529f8574734946361c461b024d99|2019-01-01 00:00:44  |2582ad8a1626ba7234d8c09bec3ad030|
|a1a22d70485983eac12b5b88dad1cf95|2019-01-01 00:00:51  |9a9a716a08b99520374fed7a129d8415|
|6b849c168bdad6f867558c3793159a81|2019-01-01 00:01:16  |89acafa4e0f4f1a99147833c3bd9e437|
|a41d7549acf90789359a9aa5346dcb46|2019-01-01 00:03:06  |b0d3d9f124cfe0b9b771e0378ba221a2|
+--------------------------------+---------------------+--------------------------------+
only showing top 5 rows


In [19]:
# Verify the key really is unique before locking it in as the dedup key.
print("rows:", silver.count())
print("distinct _scd_unique_id:", silver.select("_scd_unique_id").distinct().count())

silver.groupBy("_scd_unique_id").count().filter("count > 1").show()


rows: 1296675


distinct _scd_unique_id: 1296675


+--------------+-----+
|_scd_unique_id|count|
+--------------+-----+
+--------------+-----+



In [20]:
# Dedup on _scd_unique_id. Note the semantics: partitioning on the
# composite key keeps one row per (trans_num, timestamp) pair, whereas
# partitioning on trans_num alone would keep one row per transaction
# regardless of timestamp.
# row_number() over a window rather than dropDuplicates() — deterministic, and
# it chooses which record survives (latest ingest wins).
w = Window.partitionBy("_scd_unique_id").orderBy(F.col("_ingested_at").desc())

silver = (silver
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn"))

print("rows after:", silver.count())

rows after: 1296675


### ✍️ Dedup strategy

Check with team that providing source if there is unique id else asking if the combination of trans_num and trans_date_trans_time are unique. Creating the unique key to avoid duplication ingested into silver layer if rerun pipeline is needed.

---
# 3. Data quality gate

Rules live in `dq_rules.yaml` (dbt `schema.yml` shape) so they can change
without touching pipeline code.

This runs **before** encryption — once `first_name` is ciphertext, "does this
look like a name" stops working.

In [21]:
import yaml


def run_dq(df, model_name, config_path=DQ_RULES):
    cfg = yaml.safe_load(open(config_path))[model_name]
    total = df.count()
    results = []

    for check in cfg["checks"]:
        col = check["column"]
        tests = check.get("tests", [])
        sev = check.get("severity", "warn")

        conds = []
        if "not_null" in tests:
            conds.append((f"{col}__not_null", F.col(col).isNotNull()))
        if "accepted_values" in check:
            conds.append((f"{col}__accepted_values", F.col(col).isin(check["accepted_values"])))
        if "expression" in check:
            conds.append((f"{col}__expression", F.expr(check["expression"])))

        for name, cond in conds:
            # Three-valued logic: a null makes `cond` null, not false, so it
            # slips past a plain ~cond. Count it as a failure explicitly.
            failed = df.filter(~cond | cond.isNull()).count()
            results.append((name, sev, failed, round(100 * failed / total, 3)))

        if "unique" in tests:
            dupes = df.groupBy(col).count().filter("count > 1").count()
            results.append((f"{col}__unique", sev, dupes, round(100 * dupes / total, 3)))

    report = spark.createDataFrame(
        results, "check string, severity string, failed_rows long, failed_pct double")

    errors = [r for r in results if r[1] == "error" and r[2] > 0]
    if errors:
        report.show(50, False)
        raise ValueError(f"DQ gate failed on {len(errors)} error-severity checks")

    return report

In [22]:
dq_report = run_dq(silver, "silver_transactions")
dq_report.show(50, truncate=False)

+----------------------------------+--------+-----------+----------+
|check                             |severity|failed_rows|failed_pct|
+----------------------------------+--------+-----------+----------+
|_scd_unique_id__not_null          |error   |0          |0.0       |
|_scd_unique_id__unique            |error   |0          |0.0       |
|trans_num__not_null               |error   |0          |0.0       |
|trans_num__unique                 |error   |0          |0.0       |
|amt__not_null                     |error   |0          |0.0       |
|amt__expression                   |error   |0          |0.0       |
|gender__accepted_values           |warn    |0          |0.0       |
|is_fraud__accepted_values         |error   |0          |0.0       |
|cust_lat__expression              |warn    |0          |0.0       |
|cust_long__expression             |warn    |0          |0.0       |
|state__expression                 |warn    |0          |0.0       |
|zip__expression                  

In [23]:
# Persist every run so a check drifting from 0.1% to 4% is visible, not just pass/fail.
(dq_report
 .withColumn("model", F.lit("silver_transactions"))
 .withColumn("run_ts", F.current_timestamp())
 .write.mode("append").parquet(DQ_RESULTS))

### ✍️ Data quality framework

Using the yaml file logic rto control on data quality check so its ease for other analyst team or end user to understand and amend on the logic checking.


---
# 4. PII — encrypt for recovery, hash for joins

### What is protected, and what is not

| Column | Treatment |
|---|---|
| `cc_num` | hashed **and** encrypted |
| `cc_bic` | hashed **and** encrypted |
| `first_name`, `last_name` | encrypted |


In [24]:
# Generate the salt and key ONCE, then keep them. Regenerating the salt changes
# every hash and breaks joins to already-written data.
# import secrets
# print("PII_SALT   =", secrets.token_hex(32))
# print("PII_AES_KEY=", secrets.token_hex(16))   # 32 chars = 32 bytes, a valid AES key

In [25]:
from dotenv import load_dotenv

load_dotenv()
SALT    = os.environ["PII_SALT"]
AES_KEY = os.environ["PII_AES_KEY"]

### 4.1 Encrypt (reversible)

In [26]:
def encrypt_pii(col_name, key=AES_KEY):
    return F.when(
        F.col(col_name).isNotNull(),
        F.aes_encrypt(F.col(col_name).cast("string"), F.lit(key), F.lit("ECB"))
    )

silver = (silver
    .withColumn("cc_num_enc",     encrypt_pii("cc_num"))
    .withColumn("cc_bic_enc",     encrypt_pii("cc_bic"))
    .withColumn("first_name_enc", encrypt_pii("first_name"))
    .withColumn("last_name_enc",  encrypt_pii("last_name")))


### 4.2 Hash (irreversible) — join and grouping keys

In [27]:
def hash_pii(col_name, salt=SALT):
    return F.when(
        F.col(col_name).isNotNull(),
        F.sha2(F.concat(F.lit(salt), F.col(col_name).cast("string")), 256)
    )

silver = (silver
    .withColumn("cc_num",     hash_pii("cc_num"))
    .withColumn("cc_bic", hash_pii("cc_bic")))

### 4.3 Drop the plaintext and write silver

In [28]:

silver = silver.drop(
    "person_name", "name_tokens", "first_name", "last_name",
    "_source_file",
)

silver.printSchema()
silver.write.mode("overwrite").parquet(SILVER_PATH)


root
 |-- trans_num: string (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- cc_bic: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: decimal(12,2) (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- merch_zipcode: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- cust_lat: double (nullable = true)
 |-- cust_long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- name_parse_status: string (nullable = false)
 |-- _scd_unique_id: string (nullable = false)
 |-- cc_num_e

---
# 5. Gold — analysis tables

Three tables, split across **two access tiers**. The tier, not the transform, is
what actually protects anything.

| Table | Tier | Grain |
|---|---|---|
| `gold_decrypt_customer_master` | **restricted** — needs the AES key | one row per customer |
| `gold_analysis_customer_demographic` | analytics | one row per demographic segment |

Only the first needs `AES_KEY`. The other two read hashed/plaintext silver and
never touch the key — which is what lets them live in the general analytics
prefix.

In [29]:
silver_tbl = spark.read.parquet(SILVER_PATH)

# Age is computed against a fixed reference date, not current_date(), so the
# tables are reproducible — otherwise every re-run shifts the age bands.
ANALYSIS_DATE = F.lit("2020-07-01").cast("date")

# Customer grain = the cc_num hash, latest transaction wins. Defined here rather
# than inside 5.1 so the analytics tables can use it without the restricted cell.
w_cust = Window.partitionBy("cc_num").orderBy(F.col("trans_date_trans_time").desc())

print("silver rows:", silver_tbl.count())

silver rows: 1296675


## 5.1 `gold_decrypt_customer_master` — RESTRICTED

In [30]:
def decrypt_pii(col_name, key=AES_KEY):
    # aes_decrypt returns binary, hence the cast. A wrong key throws rather than
    # returning garbage, so a botched key rotation fails loudly.
    return F.aes_decrypt(F.col(col_name), F.lit(key), F.lit("ECB")).cast("string")


# One row per customer: latest transaction wins (w_cust defined above).
# last seen used , useful for marketing team as customer list 
gold_decrypt_customer_master = (silver_tbl
    .withColumn("_rn", F.row_number().over(w_cust))
    .filter("_rn = 1")
    .select(
        F.col("cc_num").alias("cc_num_hash"),          # stable customer key
        decrypt_pii("cc_num_enc").alias("cc_num"),
        decrypt_pii("cc_bic_enc").alias("cc_bic"),
        decrypt_pii("first_name_enc").alias("first_name"),
        decrypt_pii("last_name_enc").alias("last_name"),
        "gender", "job", "dob",
        "street", "city", "state", "zip",
        "cust_lat", "cust_long", "city_pop",
    ))

print("customers:", gold_decrypt_customer_master.count())
gold_decrypt_customer_master.show(5, truncate=False)

customers: 983


+----------------------------------------------------------------+-------------------+-----------+----------+---------+------+-------------------------------+----------+-------------------------------+-----------+-----+-----+------------------+---------+--------+
|cc_num_hash                                                     |cc_num             |cc_bic     |first_name|last_name|gender|job                            |dob       |street                         |city       |state|zip  |cust_lat          |cust_long|city_pop|
+----------------------------------------------------------------+-------------------+-----------+----------+---------+------+-------------------------------+----------+-------------------------------+-----------+-----+-----+------------------+---------+--------+
|000089d0337d3bbc60438826040e025f336f899f55e4abf74c5fb54a72060178|4911818930706644725|DEUTUS33TRF|Jeremy    |Chavez   |M     |Landscape architect            |1940-11-08|01770 Kevin Lodge Suite 190    |Mesa   

## 5.2 `gold_analysis_customer_demographic`

Aggregating transactions directly would weight each customer by how often they shop, so a
heavy spender would count as many customers.

In [32]:
# Customer-level base (no decryption — hash key + plaintext attributes only).
cust_base = (silver_tbl
    .withColumn("_rn", F.row_number().over(w_cust))
    .filter("_rn = 1")
    .select(F.col("cc_num").alias("cc_num_hash"),
            "gender", "job", "dob", "city", "state", "zip", "city_pop"))

age = F.floor(F.months_between(ANALYSIS_DATE, F.col("dob")) / 12)

cust_base = cust_base.withColumn("age_band",
    F.when(age < 25, "18-24")
     .when(age < 35, "25-34")
     .when(age < 45, "35-44")
     .when(age < 55, "45-54")
     .when(age < 65, "55-64")
     .when(age >= 65, "65+")
     .otherwise("unknown"))

In [33]:
# Spend / fraud behaviour per customer, joined onto the demographic attributes.
# built booth for promo card booth , count 
# job , can used for other credit card company for customer segment

cust_activity = (silver_tbl.groupBy(F.col("cc_num").alias("cc_num_hash"))
    .agg(F.count("*").alias("txn_count"),
         F.sum("amt").alias("total_amt"),
         F.sum("is_fraud").alias("fraud_txn_count")))

gold_analysis_customer_demographic = (cust_base.join(cust_activity, "cc_num_hash", "left")
    .groupBy("gender", "age_band", "state", "job")
    .agg(F.count("*").alias("customer_count"),
         F.sum("txn_count").alias("txn_count"),
         F.round(F.sum("total_amt"), 2).alias("total_amt"),
         F.round(F.avg("total_amt"), 2).alias("avg_amt_per_customer"),
         F.sum("fraud_txn_count").alias("fraud_txn_count"))
    .withColumn("fraud_rate_pct",
                F.round(100 * F.col("fraud_txn_count") / F.col("txn_count"), 3))
    # k-anonymity flag: a segment matching very few customers can re-identify
    # them, especially since job + state + age_band is close to unique.
    .withColumn("small_segment", F.col("customer_count") < 5)
    .orderBy(F.desc("customer_count")))

gold_analysis_customer_demographic.show(10, truncate=False)

+------+--------+-----+----------------------------------+--------------+---------+---------+--------------------+---------------+--------------+-------------+
|gender|age_band|state|job                               |customer_count|txn_count|total_amt|avg_amt_per_customer|fraud_txn_count|fraud_rate_pct|small_segment|
+------+--------+-----+----------------------------------+--------------+---------+---------+--------------------+---------------+--------------+-------------+
|F     |65+     |PA   |Mining engineer                   |2             |4091     |260519.63|130259.82           |14             |0.342         |true         |
|M     |35-44   |TX   |Petroleum engineer                |2             |2548     |244324.38|122162.19           |14             |0.549         |true         |
|M     |55-64   |WI   |Public relations officer          |1             |518      |31084.88 |31084.88            |5              |0.965         |true         |
|F     |45-54   |WA   |Public house mana

In [36]:
tot = gold_analysis_customer_demographic.count()
small = gold_analysis_customer_demographic.filter("small_segment").count()
print(f"segments: {tot} | segments with <5 customers: {small} ({round(100*small/tot,1)}%)")

gold_analysis_customer_demographic.write.mode("overwrite").parquet(f"{GOLD_PATH}/analysis_customer_demographic")

segments: 981 | segments with <5 customers: 981 (100.0%)
